# 🤖 JSON AI Tool Router & Validator

A practical project that turns **JSON, JSON Schema, AI tool definitions, parameters, and tool registries** into a working mini AI Tool Router.

**Flow:** JSON Request → Schema Validation → Tool Registry → Router → Python Tool → Structured JSON Result 🚀

![Architecture](architecture.png)


## 1️⃣ Define the JSON Tool Catalog

In [ ]:
import json

tool_catalog = {
    "namespace": "support",
    "description": "Customer support tools",
    "tools": [
        {
            "name": "get_order_status",
            "description": "Get the current status of an order.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
                "additionalProperties": False
            }
        },
        {
            "name": "create_ticket",
            "description": "Create a customer support ticket.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"},
                    "issue": {"type": "string"}
                },
                "required": ["customer_id", "issue"],
                "additionalProperties": False
            }
        }
    ]
}
print(json.dumps(tool_catalog, indent=2))


## 2️⃣ Create Mock Tool Implementations

These deterministic mock functions require no external APIs.

In [ ]:
def get_order_status(order_id: str):
    if order_id == "ORD1001":
        return {"order_id": order_id, "status": "shipped"}
    return {"order_id": order_id, "status": "not_found"}

def create_ticket(customer_id: str, issue: str):
    return {
        "ticket_id": "TCK-1001",
        "customer_id": customer_id,
        "issue": issue,
        "status": "created"
    }

TOOL_IMPLEMENTATIONS = {
    "get_order_status": get_order_status,
    "create_ticket": create_ticket
}


## 3️⃣ Build a Lightweight JSON Schema Validator

In [ ]:
def validate_arguments(schema, arguments):
    if schema.get("type") != "object":
        return False, "Root schema must be an object."

    required = schema.get("required", [])
    missing = [key for key in required if key not in arguments]
    if missing:
        return False, f"Missing required fields: {missing}"

    if schema.get("additionalProperties") is False:
        allowed = set(schema.get("properties", {}))
        extra = [key for key in arguments if key not in allowed]
        if extra:
            return False, f"Unexpected fields: {extra}"

    for key, definition in schema.get("properties", {}).items():
        if key not in arguments:
            continue
        if definition.get("type") == "string" and not isinstance(arguments[key], str):
            return False, f"{key} must be a string."

    return True, "Valid arguments."


## 4️⃣ Build the Tool Registry

In [ ]:
TOOL_REGISTRY = {
    tool["name"]: {
        "schema": tool["parameters"],
        "implementation": TOOL_IMPLEMENTATIONS[tool["name"]],
        "description": tool["description"]
    }
    for tool in tool_catalog["tools"]
}

print(list(TOOL_REGISTRY))


## 5️⃣ Implement the JSON Tool Router

In [ ]:
def route_tool_call(request):
    tool_name = request.get("name")
    arguments = request.get("arguments", {})

    if tool_name not in TOOL_REGISTRY:
        return {"ok": False, "error": f"Unknown tool: {tool_name}"}

    entry = TOOL_REGISTRY[tool_name]
    valid, message = validate_arguments(entry["schema"], arguments)

    if not valid:
        return {"ok": False, "error": message}

    result = entry["implementation"](**arguments)
    return {"ok": True, "tool": tool_name, "result": result}


## 6️⃣ Test a Valid Tool Call

In [ ]:
request = {
    "name": "get_order_status",
    "arguments": {"order_id": "ORD1001"}
}
print(json.dumps(route_tool_call(request), indent=2))


## 7️⃣ Test Validation Failures

In [ ]:
bad_requests = [
    {"name": "get_order_status", "arguments": {}},
    {"name": "get_order_status", "arguments": {"order_id": 1001}},
    {"name": "get_order_status", "arguments": {"order_id": "ORD1001", "debug": True}},
    {"name": "unknown_tool", "arguments": {}}
]

for item in bad_requests:
    print(json.dumps(route_tool_call(item), indent=2))
    print("-" * 50)


## 8️⃣ Final Exercise 🚀

Add a `refund_order` tool.

Requirements:
- `order_id`: required string
- `reason`: required string
- Reject additional properties
- Add a Python implementation
- Register it
- Route a valid request
- Test at least two invalid requests

## 🧠 What This Project Teaches

| Concept | Practical role |
|---|---|
| JSON | Tool metadata and requests |
| JSON Schema | Input contract |
| `required` | Mandatory arguments |
| `additionalProperties` | Reject unexpected arguments |
| Tool registry | Maps names to implementations |
| Router | Selects the requested tool |
| Structured result | Predictable tool output |

**Mental model:** `JSON Schema → validate → registry lookup → execute → structured JSON result`
